# Del dato a la decisión: factor de rendimiento del café en una trilladora de Pitalito (Huila)

**Proyecto de aula · Corte 3 · Electiva VIII · Ciencia de Datos (Cód. 69109) · CORHUILA · 2026-B**
**Estudiante:** Cristhian Gaitán Guzmán (individual) · **Docente:** Jesús Ariel González Bonilla

> Empresa **ficticia** (nombre inventado) con datos **sintéticos pero coherentes**, generados en este proyecto. Cualquier parecido con una empresa real es coincidencia.

| Fase | Revisión | Partes del cuaderno | Estado |
|---|---|---|---|
| 0 · Definición | sáb 31-oct | 1 | ✅ completa |
| 1 · Datos, limpieza y consultas | sáb 7-nov | 2 a 5 | ⏳ pendiente |
| 2 · Análisis, tablero, historia y sustentación | sáb 14-nov | 0, 6 a 9 | ⏳ pendiente |

## Parte 0 · Overview & insights  *(se escribe al final, pero va primero)*

**Qué hace esta parte:** resume el proyecto completo **en inglés**, con **mínimo seis oraciones**, una por cada idea de la lista. Se redacta cuando ya están los resultados (Fase 2), pero se deja aquí porque es lo primero que lee el evaluador.

**Estado: pendiente (Fase 2).** Plantilla:

1. **Problem:** *(what decision the company makes poorly today and why)*
2. **Data:** *(what data was used, how many rows, which period, synthetic)*
3. **Cleaning:** *(which quality problems were found and how they were fixed, with numbers)*
4. **Analysis:** *(what was measured: KPI by month, moving average, period comparison)*
5. **Key insight:** *(the main finding, with numbers)*
6. **Recommendation:** *(what the company should do, who, and when)*

## Parte 1 · Ficha del proyecto  *(Fase 0)*

**Qué se hace y por qué:** antes de tocar datos hay que dejar claro **qué decisión** se quiere mejorar, **qué pregunta** la responde y **cómo se mide**. La ficha fija esos acuerdos para que las demás partes del cuaderno (limpieza, consultas, tablero) no se desvíen. Es la evidencia de la Fase 0.

In [1]:
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)

ficha = {
    "Integrantes (máximo dos)": "Cristhian Gaitán Guzmán (proyecto individual)",
    "Empresa y tipo (real o ficticia)": "Trilladora Cumbres del Macizo S.A.S. · FICTICIA (nombre inventado), ubicada en Pitalito, Huila",
    "A qué se dedica": (
        "Compra café pergamino seco a caficultores y asociaciones del sur del Huila, lo trilla (retira la cascarilla), "
        "lo clasifica en excelso y pasilla y lo vende a exportadoras. Recibe y trilla unos 150 lotes al mes."
    ),
    "Problema": (
        "Cada lote que llega se pesa, se mide su humedad y, al trillarlo, se obtiene su factor de rendimiento "
        "(kg de pergamino necesarios para un saco de 70 kg de excelso). Pero esos registros quedan en archivos "
        "sueltos y nunca se consolidan. Sin esa mirada histórica, la empresa decide a qué proveedores pedirles mejorar el "
        "secado, cuándo reforzar el control de humedad en la recepción y a quién priorizar al comprar por costumbre, "
        "y no sabe qué proveedores ni qué meses concentran el café que rinde peor que la referencia de 94 kg."
    ),
    "Pregunta": (
        "¿Qué proveedores y qué meses, entre octubre de 2025 y septiembre de 2026, entregaron café pergamino "
        "con peor factor de rendimiento?"
    ),
    "KPI principal y fórmula": (
        "Factor de rendimiento ponderado = 70 × Σ kg de pergamino trillado ÷ Σ kg de excelso obtenido. "
        "Se interpreta como kg de pergamino por cada saco de 70 kg de excelso: menor es mejor; referencia = 94. "
        "Se calcula con totales por proveedor y por mes, nunca promediando los factores de cada lote."
    ),
    "KPI de apoyo": (
        "1) % de lotes con factor mayor a 94 = lotes con factor > 94 ÷ total de lotes × 100.  "
        "2) % de lotes con humedad fuera del rango 10 % a 12 % = lotes fuera de rango ÷ total de lotes × 100."
    ),
    "Fuentes de datos y formato": (
        "1) lotes_recepcion.csv: exportación simulada del software de recepción, una fila por lote (≥ 1.000 filas, oct-2025 a sep-2026). "
        "2) proveedores.xlsx: maestro de proveedores en Excel (código, municipio, tipo de proveedor). "
        "Ambos se generan con Python (semilla fija, reglas documentadas) en la Fase 1, con problemas de calidad a propósito."
    ),
    "Autorización (solo empresa real)": "No aplica: empresa ficticia",
}

tabla_ficha = pd.DataFrame({"Campo": list(ficha.keys()), "Su respuesta": list(ficha.values())})
display(tabla_ficha.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
print(f"Campos de la ficha completados: {len(ficha)} de 9")

Campo,Su respuesta
Integrantes (máximo dos),Cristhian Gaitán Guzmán (proyecto individual)
Empresa y tipo (real o ficticia),"Trilladora Cumbres del Macizo S.A.S. · FICTICIA (nombre inventado), ubicada en Pitalito, Huila"
A qué se dedica,"Compra café pergamino seco a caficultores y asociaciones del sur del Huila, lo trilla (retira la cascarilla), lo clasifica en excelso y pasilla y lo vende a exportadoras. Recibe y trilla unos 150 lotes al mes."
Problema,"Cada lote que llega se pesa, se mide su humedad y, al trillarlo, se obtiene su factor de rendimiento (kg de pergamino necesarios para un saco de 70 kg de excelso). Pero esos registros quedan en archivos sueltos y nunca se consolidan. Sin esa mirada histórica, la empresa decide a qué proveedores pedirles mejorar el secado, cuándo reforzar el control de humedad en la recepción y a quién priorizar al comprar por costumbre, y no sabe qué proveedores ni qué meses concentran el café que rinde peor que la referencia de 94 kg."
Pregunta,"¿Qué proveedores y qué meses, entre octubre de 2025 y septiembre de 2026, entregaron café pergamino con peor factor de rendimiento?"
KPI principal y fórmula,"Factor de rendimiento ponderado = 70 × Σ kg de pergamino trillado ÷ Σ kg de excelso obtenido. Se interpreta como kg de pergamino por cada saco de 70 kg de excelso: menor es mejor; referencia = 94. Se calcula con totales por proveedor y por mes, nunca promediando los factores de cada lote."
KPI de apoyo,1) % de lotes con factor mayor a 94 = lotes con factor > 94 ÷ total de lotes × 100. 2) % de lotes con humedad fuera del rango 10 % a 12 % = lotes fuera de rango ÷ total de lotes × 100.
Fuentes de datos y formato,"1) lotes_recepcion.csv: exportación simulada del software de recepción, una fila por lote (≥ 1.000 filas, oct-2025 a sep-2026). 2) proveedores.xlsx: maestro de proveedores en Excel (código, municipio, tipo de proveedor). Ambos se generan con Python (semilla fija, reglas documentadas) en la Fase 1, con problemas de calidad a propósito."
Autorización (solo empresa real),No aplica: empresa ficticia


Campos de la ficha completados: 9 de 9


### Por qué el KPI se calcula con totales (y no promediando factores)

El factor de cada lote pesa distinto: un lote de 1.000 kg no puede valer lo mismo que uno de 100 kg. Con dos lotes de ejemplo se ve la diferencia. *(Es la misma idea de la Semana 9: la tasa se calcula con totales.)*

In [2]:
def factor_rendimiento(kg_pergamino, kg_excelso):
    """kg de pergamino seco necesarios para obtener un saco de 70 kg de excelso."""
    return 70 * kg_pergamino / kg_excelso

lotes_ejemplo = pd.DataFrame({"lote": ["A", "B"], "kg_pergamino": [1000, 100], "kg_excelso": [760, 70]})
factor_lote = factor_rendimiento(lotes_ejemplo["kg_pergamino"], lotes_ejemplo["kg_excelso"])   # sin redondear para promediar
lotes_ejemplo["factor"] = factor_lote.round(2)

promedio_simple = factor_lote.mean()
ponderado = factor_rendimiento(lotes_ejemplo["kg_pergamino"].sum(), lotes_ejemplo["kg_excelso"].sum())

display(lotes_ejemplo)
print(f"Promedio simple de los factores: {promedio_simple:.2f}  (engañoso: el lote pequeño pesa igual que el grande)")
print(f"Factor ponderado con totales   : {ponderado:.2f}  (el que usa el KPI)")

# Comprobaciones: la referencia de 94 kg y el cálculo del ejemplo
assert abs(factor_rendimiento(94, 70) - 94) < 1e-9           # 94 kg de pergamino para 70 kg de excelso = factor 94
assert round(promedio_simple, 2) == 96.05 and round(ponderado, 2) == 92.77

,lote,kg_pergamino,kg_excelso,factor
0,A,1000,760,92.11
1,B,100,70,100.00


Promedio simple de los factores: 96.05  (engañoso: el lote pequeño pesa igual que el grande)
Factor ponderado con totales   : 92.77  (el que usa el KPI)


## Parte 2 · Carga de datos

**Qué se hace y por qué:** leer `lotes_recepcion.csv` y `proveedores.xlsx`, mostrar su **tamaño (filas y columnas)** y una **muestra** de cada una, para saber con qué se trabaja antes de limpiar.

**Estado: pendiente (Fase 1).** Incluye también la descripción de cómo se generaron los datos sintéticos (reglas y semilla).

## Parte 3 · Diagnóstico de calidad

**Qué se hace y por qué:** medir los problemas de cada fuente **con cifras** (vacíos, duplicados, fechas en formatos distintos, textos inconsistentes, valores imposibles), porque sin el «antes» no se puede demostrar el «después».

**Estado: pendiente (Fase 1).**

## Parte 4 · Limpieza

**Qué se hace y por qué:** corregir **al menos tres problemas**, cada uno con su **antes, su después y el motivo** de la decisión (eliminar, imputar o corregir), sobre una copia de trabajo, y cerrar con una **bitácora** de cambios.

**Estado: pendiente (Fase 1).**

## Parte 5 · Base SQLite y consultas

**Qué se hace y por qué:** documentar los datos limpios (**diccionario** y **grano** de la tabla principal), cargarlos en SQLite y responder partes de la pregunta con **tres consultas o más**: una con `GROUP BY`, una con `JOIN` y una que filtre por fechas, aplicando las buenas prácticas de la Semana 14.

**Estado: pendiente (Fase 1).**

## Parte 6 · Análisis temporal

**Qué se hace y por qué:** ver cómo cambia el KPI en el tiempo: **KPI por mes**, **promedio móvil** para la tendencia y **comparación entre periodos** (primer frente a segundo semestre).

**Estado: pendiente (Fase 2).**

## Parte 7 · Tablero

**Qué se hace y por qué:** reunir en una sola vista la **fila de KPI** y **al menos cuatro gráficos**, cada uno con un título que diga el hallazgo y no solo el nombre de la variable.

**Estado: pendiente (Fase 2).**

## Parte 8 · Cifras de control

**Qué se hace y por qué:** dejar los valores que permiten **verificar el trabajo**: filas antes y después de la limpieza, el valor del KPI principal y dos totales que cualquiera pueda recalcular.

**Estado: pendiente (Fase 2).**

## Parte 9 · Historia y recomendación

**Qué se hace y por qué:** convertir el análisis en una decisión: **contexto → hallazgo → recomendación accionable** (qué debe hacer la empresa, quién y cuándo).

**Estado: pendiente (Fase 2).**